# Local Iceberg Lakehouse — Demo

**Local Iceberg Lakehouse** is a "local-first" data lakehouse: it gives you real [Apache Iceberg](https://iceberg.apache.org/) tables — ACID transactions, schema tracking, snapshots, time travel, rollbacks — running entirely on your own machine, with no cloud object storage (S3/MinIO) and no Spark cluster required.

**The stack:**

| Layer | Technology |
|---|---|
| Table format | **Apache Iceberg** (via `pyiceberg`) |
| Catalog | **SQLite** (`pyiceberg`'s SQL catalog — a single `catalog.db` file) |
| Data files | **Parquet**, written to the local filesystem warehouse (`~/.lakehouse/warehouse` by default) |
| Query engine | **DuckDB**, bridged to Iceberg scans via **Apache Arrow** |
| Interfaces | A `click`-based **CLI** (`lakehouse ...`) and an **MCP server** (`local_iceberg_lakehouse.server`) so LLMs like Claude can query/upsert data directly |

**The problem it solves:** normally, trying out Iceberg means standing up a Hive/REST/Glue catalog plus S3-compatible object storage plus (often) a Spark cluster just to run a toy example. This project swaps every remote piece for a local equivalent — SQLite instead of a hosted catalog service, the local filesystem instead of S3, DuckDB instead of Spark — so you get a fully working, spec-compliant Iceberg lakehouse for local development, testing, or even day-to-day personal data analytics, with zero external services.

This notebook walks through the real modules in `src/local_iceberg_lakehouse/` — `catalog.py`, `query.py`, `cli.py`, `server.py` — and exercises them directly (not reimplemented/mocked) to create a table, write and query data, run an upsert, and use Iceberg's time-travel/rollback features.

## Architecture

From `README.md`:

```text
┌─────────────────────────────────────────────────────────────────┐
│          LLM (Claude) / CLI Interface                           │
│          "Query my monthly expenses..."                         │
└─────────────────────────┬───────────────────────────────────────┘
                          │ MCP Protocol / CLI Commands
                          ▼
┌─────────────────────────────────────────────────────────────────┐
│          MCP Server / CLI Logic                                 │
│          Tools: query, upsert, list_tables, etc.                │
└─────────────────────────┬───────────────────────────────────────┘
                          │
                          ▼
┌─────────────────────────────────────────────────────────────────┐
│          DuckDB (Query Engine)                                  │
│          (In-memory execution with Arrow bridge)                │
└───────────────┬─────────────────────────┬───────────────────────┘
                │ PyIceberg               │ Arrow
                ▼                         ▼
┌──────────────────────────┐      ┌────────────────────────────────────┐
│      Iceberg Catalog     │      │          Data Files                │
│       (SQLite DB)        │      │       (Standard Parquet)           │
└──────────────────────────┘      └────────────────────────────────────┘
```

### How it maps to the code

- **`local_iceberg_lakehouse.catalog.CatalogManager`** (`src/local_iceberg_lakehouse/catalog.py`) wraps `pyiceberg.catalog.load_catalog(...)`, configured with `type="sql"`, a `sqlite:///.../catalog.db` URI, and a `file://` warehouse path (defaults to `~/.lakehouse/warehouse`). It exposes `create_table`, `load_table`, `list_tables`, `drop_table`, and `table_exists` — thin, direct wrappers around the PyIceberg catalog.
- **`local_iceberg_lakehouse.query.QueryEngine`** (`src/local_iceberg_lakehouse/query.py`) owns an in-memory DuckDB connection (`duckdb.connect(":memory:")`) and:
  - `query(sql, table_mapping)` — validates the SQL is a single read-only `SELECT`/`WITH` statement (see `_validate_read_only_sql`), scans the requested Iceberg table(s) to Arrow (`table.scan().to_arrow()`), registers them as DuckDB views, and runs the SQL.
  - `append_data(table_name, data)` — `table.append(data)` (a new Iceberg snapshot/append commit).
  - `overwrite_data(table_name, data)` — `table.overwrite(data)`.
  - `upsert_data(table_name, data, join_cols)` — the "Load-Merge-Overwrite" strategy described in the README: load existing data to Arrow, dedupe the incoming batch on `join_cols` (last row wins) using a DuckDB window function, `UNION ALL` new rows with non-matching existing rows, then `table.overwrite(...)` the merged result.
- **`local_iceberg_lakehouse.cli`** (`src/local_iceberg_lakehouse/cli.py`) exposes this as the `lakehouse` command (`list-tables`, `query`, `create-sample-table`, `start-server`) via `click`.
- **`local_iceberg_lakehouse.server`** (`src/local_iceberg_lakehouse/server.py`) exposes the same functionality as MCP tools (`list_tables`, `get_table_schema`, `query`, `upsert`, `rollback`, `get_history`) via `mcp.server.fastmcp.FastMCP`, so an LLM client (e.g. Claude Desktop) can call them directly. `rollback` and `get_history` use PyIceberg's native snapshot APIs: `table.manage_snapshots().rollback_to_snapshot(snapshot_id).commit()` and `table.snapshots()`.

### Setup

Dependencies come from `pyproject.toml` (`click`, `duckdb`, `mcp`, `pyarrow`, `pyiceberg[sql-sqlite]`, `sqlalchemy`, `pandas`). Per the README:

```bash
git clone https://github.com/thompgt/local-iceberg-lakehouse.git
cd local-iceberg-lakehouse
uv sync            # or: pip install -e .
```

This sandbox has `pyiceberg`, `duckdb`, and `pyarrow` installed (via `pip install -e .`), so the cells below actually execute against a real, temporary local Iceberg warehouse — not mocked output. If you're reading this without having run `uv sync` / `pip install -e .` first, the code is still the real API; it just won't have anything to import.

## 1. Set up an isolated local warehouse

`CatalogManager` defaults to `~/.lakehouse/warehouse`, which is what the CLI and MCP server use. For this demo we point it at a throwaway temp directory instead (via the `warehouse_path` constructor argument, exactly as `tests/test_catalog.py` and `tests/test_query.py` do with their `temp_warehouse` / `lakehouse` pytest fixtures) so the notebook doesn't touch — or depend on — any real lakehouse data on this machine.

In [1]:
import tempfile
from pathlib import Path

from local_iceberg_lakehouse.catalog import CatalogManager
from local_iceberg_lakehouse.query import QueryEngine

demo_warehouse = Path(tempfile.mkdtemp(prefix="iceberg_demo_"))
print("Demo warehouse path:", demo_warehouse)

cm = CatalogManager(catalog_name="demo", warehouse_path=str(demo_warehouse))
qe = QueryEngine(cm)

print("Catalog DB created:", (demo_warehouse / "catalog.db").exists())

Demo warehouse path: C:\Users\thoma\AppData\Local\Temp\iceberg_demo_8_yzb0s0


Catalog DB created: True


## 2. Create an Iceberg table

`CatalogManager.create_table` takes a `pyiceberg.schema.Schema` built from `NestedField`s — the same pattern used in `cli.py`'s `create-sample-table` command and throughout `tests/test_catalog.py` / `tests/test_query.py`. We'll create a `default.people` table, mirroring the CLI's sample table.

In [2]:
from pyiceberg.schema import Schema
from pyiceberg.types import LongType, NestedField, StringType

schema = Schema(
    NestedField(field_id=1, name="id", field_type=LongType(), required=False),
    NestedField(field_id=2, name="name", field_type=StringType(), required=False),
)

table_name = "default.people"
table = cm.create_table(table_name, schema)

print("Created:", table_name)
print("Exists?", cm.table_exists(table_name))
print("Tables in 'default' namespace:", cm.list_tables())

Created: default.people
Exists? True
Tables in 'default' namespace: ['default.people']


## 3. Write sample data (append)

`QueryEngine.append_data(table_name, data)` calls `table.append(data)` under the hood, committing a new Iceberg snapshot. We build a small `pandas` DataFrame, convert it to an Arrow table (what the Iceberg/PyIceberg APIs expect), and append it — exactly the flow in `tests/test_query.py::test_append_and_query`.

In [3]:
import pandas as pd
import pyarrow as pa

people_df = pd.DataFrame({
    "id": [1, 2, 3],
    "name": ["Alice", "Bob", "Charlie"],
})

arrow_table = pa.Table.from_pandas(people_df)
qe.append_data(table_name, arrow_table)

print(f"Appended {len(people_df)} rows to {table_name}")

Appended 3 rows to default.people


## 4. Query it back with DuckDB

`QueryEngine.query(sql, table_mapping)` scans the Iceberg table to Arrow and registers it as a DuckDB view under the logical name given in `table_mapping`, then runs the SQL — the same pattern used by the `lakehouse query` CLI command and the MCP `query` tool. Note it only allows a single read-only `SELECT`/`WITH` statement (`_validate_read_only_sql`); mutating SQL (`INSERT`, `ATTACH`, `COPY`, etc.) is rejected, as covered by `tests/test_query.py::test_query_rejects_non_read_only_sql`.

In [4]:
result = qe.query(
    "SELECT * FROM people ORDER BY id",
    table_mapping={"people": table_name},
)
result.to_pandas()

,id,name
0,1,Alice
1,2,Bob
2,3,Charlie


## 5. Upsert (Load-Merge-Overwrite)

`QueryEngine.upsert_data(table_name, data, join_cols)` implements the strategy documented in the README's "Technical Design Decisions": load existing data, merge the new batch in DuckDB (dedup on `join_cols`, then `UNION ALL` new rows with non-matching existing rows), and `table.overwrite(...)` the merged result — one new Iceberg snapshot replacing the table's contents. This mirrors `tests/test_query.py::test_upsert`: we update Bob's name and add Diana.

In [5]:
upsert_df = pd.DataFrame({
    "id": [2, 4],
    "name": ["Bobby", "Diana"],
})

qe.upsert_data(table_name, pa.Table.from_pandas(upsert_df), join_cols=["id"])

qe.query("SELECT * FROM people ORDER BY id", table_mapping={"people": table_name}).to_pandas()

,id,name
0,1,Alice
1,2,Bobby
2,3,Charlie
3,4,Diana


## 6. Time travel and rollback

Every `append`/`overwrite` commit creates a new Iceberg **snapshot**. The MCP server's `get_history` tool lists them via `table.snapshots()`, and `rollback` reverts via `table.manage_snapshots().rollback_to_snapshot(snapshot_id).commit()` (see `src/local_iceberg_lakehouse/server.py`). We reload the table (to pick up the snapshots committed above) and use those same PyIceberg APIs directly.

In [6]:
table = cm.load_table(table_name)

snapshots = list(table.snapshots())
for s in snapshots:
    op = s.summary.operation if s.summary is not None else None
    print(f"snapshot_id={s.snapshot_id}  timestamp_ms={s.timestamp_ms}  operation={op}")

first_snapshot_id = snapshots[0].snapshot_id
print("\nWill roll back to the first snapshot (post-append, pre-upsert):", first_snapshot_id)

snapshot_id=13324792739998363  timestamp_ms=1784247752972  operation=Operation.APPEND
snapshot_id=8523665224841701343  timestamp_ms=1784247753186  operation=Operation.DELETE
snapshot_id=500586910637204473  timestamp_ms=1784247753223  operation=Operation.APPEND

Will roll back to the first snapshot (post-append, pre-upsert): 13324792739998363


In [7]:
# Time travel: read the table AS OF the first snapshot, without mutating current state.
historical_arrow = table.scan(snapshot_id=first_snapshot_id).to_arrow()
print("Data as of first snapshot (before the upsert):")
print(historical_arrow.to_pandas())

# Rollback: actually move the table's current state back to that snapshot.
table.manage_snapshots().rollback_to_snapshot(first_snapshot_id).commit()

print("\nCurrent state after rollback:")
qe.query("SELECT * FROM people ORDER BY id", table_mapping={"people": table_name}).to_pandas()

Data as of first snapshot (before the upsert):
   id     name
0   1    Alice
1   2      Bob
2   3  Charlie

Current state after rollback:


,id,name
0,1,Alice
1,2,Bob
2,3,Charlie


## 7. CLI and MCP server (for reference)

The same `CatalogManager`/`QueryEngine` operations are exposed two other ways in this project:

**CLI** (`local_iceberg_lakehouse.cli:cli`, installed as the `lakehouse` script):

```bash
uv run lakehouse create-sample-table
uv run lakehouse list-tables
uv run lakehouse query default.people --sql "SELECT * FROM people WHERE id > 0"
uv run lakehouse start-server
```

`tests/test_cli.py` exercises these with Click's `CliRunner`, e.g. `runner.invoke(cli_module.cli, ["create-sample-table"])`.

**MCP server** (`local_iceberg_lakehouse.server:mcp`, a `FastMCP` app), exposing tools an LLM client can call directly: `list_tables`, `get_table_schema`, `query`, `upsert`, `rollback`, `get_history`. This is what lets an assistant like Claude Desktop manage the lakehouse via natural language, per the `claude_desktop_config.json` snippet in the README.

In [8]:
# Reuse the CLI's own `cli` Click group against our demo warehouse, the same way
# tests/test_cli.py does (it monkeypatches HOME/USERPROFILE; here we just point
# the CLI's query command at our already-created demo table via its full name).
from click.testing import CliRunner

from local_iceberg_lakehouse import cli as cli_module

runner = CliRunner()
# The CLI's `query` command builds its own CatalogManager() pointed at the default
# ~/.lakehouse/warehouse, not our temp one, so here we just show the command's
# --help to confirm the real CLI surface without touching the real warehouse.
result = runner.invoke(cli_module.cli, ["query", "--help"])
print(result.output)

Usage: cli query [OPTIONS] TABLE_NAME

  Run a query against a table.

Options:
  --sql TEXT  SQL query to run  [required]
  --help      Show this message and exit.



## Summary

This notebook exercised the real `local_iceberg_lakehouse` package end to end against a temporary local warehouse: `CatalogManager` created a genuine Iceberg table backed by a SQLite catalog and Parquet files on disk; `QueryEngine` appended a pandas/Arrow DataFrame, queried it through DuckDB, and performed a load-merge-overwrite upsert; and native PyIceberg snapshot APIs (`table.snapshots()`, `table.scan(snapshot_id=...)`, `table.manage_snapshots().rollback_to_snapshot(...)`) demonstrated time travel and rollback — the same APIs the project's MCP server exposes as `get_history` and `rollback` tools. The same functionality is also reachable via the `lakehouse` CLI (`create-sample-table`, `list-tables`, `query`, `start-server`) and via MCP for direct use by an LLM assistant like Claude — all without any cloud object storage or Spark cluster, just local files.